In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Labs"
)

xpt_file = DATA_DIR / "FASTQX_J.xpt"
csv_file = DATA_DIR / "FASTQX_J_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Completely untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2017-2018 FASTQX_J XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "PHQ020",
    "PHACOFHR",
    "PHACOFMN",
    "PHQ030",
    "PHAALCHR",
    "PHAALCMN",
    "PHQ040",
    "PHAGUMHR",
    "PHAGUMMN",
    "PHQ050",
    "PHAANTHR",
    "PHAANTMN",
    "PHQ060",
    "PHASUPHR",
    "PHASUPMN",
    "PHAFSTHR",
    "PHAFSTMN",
    "PHDSESN"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "FASTQX_J columns/order differ from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 8366:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 19:

    raise ValueError(
        f"Unexpected column count: "
        f"{len(df.columns)}"
    )


print(
    "PASS: Structure = 8,366 rows x 19 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Direct inspection of FASTQX_J shows:
#
#     NO genuine fractional variables.
#
# Variables consist of:
#
#     SEQN
#     Yes/No codes
#     elapsed hours
#     elapsed minutes
#     examination session code
#
#
# Therefore all 19 fields can become nullable Int64
# AFTER proving that all released observations are integral.
#
#
# IMPORTANT:
#
# This applies ONLY to FASTQX_J.
#
# Do NOT copy this assumption to:
#
#     Dietary
#     Laboratory measurements
#     survey weights
#     nutrients
#     environmental exposure values
#     examination decimal variables
#
# Genuine decimals in those files must remain untouched.
#

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()

DECIMAL_COLUMNS = []


print("\n--- FIELD TYPE PLAN ---")

print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


# ============================================================
# 5. VERIFY ALL VARIABLES LOADED NUMERICALLY
# ============================================================

numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 19 FASTQX_J variables "
        "to load numerically.\n\n"
        f"Actual numeric columns:\n{numeric_cols}"
    )


print(
    "PASS: All 19 FASTQX_J variables "
    "loaded numerically."
)


# ============================================================
# 6. KNOWN SAS XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT decoded zero:
#
#     5.397605346934028e-79
#
#
# NEVER blindly do:
#
#     df.replace(TINY_VALUE, 0)
#
#
# We first prove exactly which variables contain it.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


tiny_counts = (
    df[EXPECTED_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[EXPECTED_COLUMNS]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Ordinary numeric zeros before repair:",
    f"{true_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "\nTiny values by variable:"
)

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 7. VALIDATED ATTACHMENT-SPECIFIC TINY PATTERN
# ============================================================

#
# Direct inspection of attached FASTQX_J.xpt:
#
# PHACOFHR       5
# PHACOFMN       2
#
# PHAALCHR       1
# PHAALCMN       2
#
# PHAGUMHR      63
# PHAGUMMN       8
#
# PHAANTHR       7
# PHAANTMN       2
#
# PHASUPHR       5
# PHASUPMN       5
#
# PHAFSTHR     325
# PHAFSTMN     146
#
# PHDSESN     3818
#
# TOTAL       4389
#
#
# CDC permits 0 in all corresponding hour/minute ranges.
#
# PHDSESN is especially strong validation:
#
#     CDC:
#         0 = morning
#         count = 3818
#
# which matches the tiny-value count exactly.
#

EXPECTED_TINY_COUNTS = {
    "PHACOFHR": 5,
    "PHACOFMN": 2,
    "PHAALCHR": 1,
    "PHAALCMN": 2,
    "PHAGUMHR": 63,
    "PHAGUMMN": 8,
    "PHAANTHR": 7,
    "PHAANTMN": 2,
    "PHASUPHR": 5,
    "PHASUPMN": 5,
    "PHAFSTHR": 325,
    "PHAFSTMN": 146,
    "PHDSESN": 3818
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: FASTQX_J tiny-value pattern "
        "differs from validated attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 4389:

    raise ValueError(
        f"Unexpected total tiny count: "
        f"{tiny_total:,}"
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros existed "
        "before XPORT correction."
    )


print(
    "PASS: Exact 4,389-value tiny-zero "
    "pattern validated."
)


# ============================================================
# 8. RESTORE ONLY THE VALIDATED ZEROS
# ============================================================

#
# Each variable is repaired independently.
#
# NO global replace.
#

correction_counts = {}


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )

    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col} tiny count changed.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_counts[col] = actual_count


print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)


for col, count in correction_counts.items():

    print(
        f"{col}: {count:,}"
    )


total_corrections = sum(
    correction_counts.values()
)


if total_corrections != 4389:

    raise ValueError(
        "Total zero restoration count incorrect."
    )


print(
    "PASS: 4,389 validated XPORT zeros restored."
)


# ============================================================
# 9. CREATE EXPECTED CORRECTED SOURCE
# ============================================================

#
# This represents exactly what the final dataset is
# permitted to contain.
#
# ONLY validated tiny values become zero.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 10. VERIFY NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[EXPECTED_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny artifacts remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 11. FRACTIONAL VALUE DISCOVERY
# ============================================================

#
# Run BEFORE Int64 conversion.
#
# np.round() is ONLY used as a comparison test.
#
# It DOES NOT modify the dataframe.
#

print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)


fractional_counts = {}


for col in EXPECTED_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_count = 0

    else:

        fractional_mask = ~np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )

        fractional_count = int(
            fractional_mask.sum()
        )


    fractional_counts[col] = (
        fractional_count
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


fractional_variables = {
    col: count
    for col, count in fractional_counts.items()
    if count > 0
}


if fractional_variables:

    raise ValueError(
        "STOP: Unexpected genuine decimal values "
        "found in FASTQX_J.\n\n"
        + str(fractional_variables)
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 12. VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractions."
    )


print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 93703:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 102956:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicates."
    )


print(
    "PASS: SEQN 93703-102956 validated."
)


# ============================================================
# 13. FREQUENCY CHECK HELPER
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- CDC CHECK: {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        actual_missing
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 14. YES/NO QUESTION FREQUENCY CHECKS
# ============================================================

check_frequency(
    "PHQ020",
    {
        1: 118,
        2: 7878
    },
    370
)


check_frequency(
    "PHQ030",
    {
        1: 39,
        2: 7957
    },
    370
)


check_frequency(
    "PHQ040",
    {
        1: 167,
        2: 7829
    },
    370
)


check_frequency(
    "PHQ050",
    {
        1: 65,
        2: 7931
    },
    370
)


check_frequency(
    "PHQ060",
    {
        1: 137,
        2: 7859
    },
    370
)


# ============================================================
# 15. SESSION VARIABLE CHECK
# ============================================================

#
# CDC:
#
# PHDSESN
#
#     0 = Morning      3818
#     1 = Afternoon    3153
#     2 = Evening      1395
#
# Missing = 0
#

check_frequency(
    "PHDSESN",
    {
        0: 3818,
        1: 3153,
        2: 1395
    },
    0
)


# ============================================================
# 16. RANGE CHECK HELPER
# ============================================================

def check_range(
    column,
    lower,
    upper,
    expected_nonmissing,
    expected_missing
):

    values = df[column]


    in_range = int(
        values.between(
            lower,
            upper,
            inclusive="both"
        )
        .sum()
    )


    missing = int(
        values.isna()
        .sum()
    )


    minimum = (
        values.min()
    )


    maximum = (
        values.max()
    )


    print(
        f"\n--- RANGE CHECK: {column} ---"
    )

    print(
        f"Expected range: {lower}-{upper}"
    )

    print(
        "Observed minimum:",
        minimum
    )

    print(
        "Observed maximum:",
        maximum
    )

    print(
        "In-range records:",
        in_range
    )

    print(
        "Missing:",
        missing
    )


    if in_range != expected_nonmissing:

        raise ValueError(
            f"{column} in-range count differs.\n"
            f"Expected: {expected_nonmissing:,}\n"
            f"Actual:   {in_range:,}"
        )


    if missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {missing:,}"
        )


    print(
        f"PASS: {column} range/count validated."
    )


# ============================================================
# 17. COFFEE / TEA FASTING TIME
# ============================================================

check_range(
    "PHACOFHR",
    0,
    17,
    118,
    8248
)


check_range(
    "PHACOFMN",
    0,
    59,
    118,
    8248
)


# ============================================================
# 18. ALCOHOL FASTING TIME
# ============================================================

check_range(
    "PHAALCHR",
    0,
    17,
    39,
    8327
)


check_range(
    "PHAALCMN",
    0,
    59,
    39,
    8327
)


# ============================================================
# 19. GUM / MINTS FASTING TIME
# ============================================================

check_range(
    "PHAGUMHR",
    0,
    16,
    167,
    8199
)


check_range(
    "PHAGUMMN",
    0,
    59,
    167,
    8199
)


# ============================================================
# 20. ANTACID / LAXATIVE FASTING TIME
# ============================================================

check_range(
    "PHAANTHR",
    0,
    16,
    65,
    8301
)


check_range(
    "PHAANTMN",
    0,
    57,
    65,
    8301
)


# ============================================================
# 21. DIETARY SUPPLEMENT FASTING TIME
# ============================================================

check_range(
    "PHASUPHR",
    0,
    16,
    137,
    8229
)


check_range(
    "PHASUPMN",
    0,
    59,
    137,
    8229
)


# ============================================================
# 22. TOTAL FOOD FAST TIME
# ============================================================

#
# CDC:
#
# PHAFSTHR:
#
#     0-38 hours
#     7996 populated
#     370 missing
#
#
# PHAFSTMN:
#
#     0-59 minutes
#     7996 populated
#     370 missing
#

check_range(
    "PHAFSTHR",
    0,
    38,
    7996,
    370
)


check_range(
    "PHAFSTMN",
    0,
    59,
    7996,
    370
)


# ============================================================
# 23. ROUTING CHECK HELPER
# ============================================================

#
# If respondent answered YES to having consumed an item
# since beginning the food fast, both corresponding
# elapsed hour and minute fields should be populated.
#

def check_yes_route(
    parent_column,
    child_column
):

    expected = (
        df[parent_column]
        .eq(1)
        .fillna(False)
        .to_numpy(
            dtype=bool
        )
    )


    actual = (
        df[child_column]
        .notna()
        .to_numpy(
            dtype=bool
        )
    )


    mismatches = int(
        np.sum(
            expected != actual
        )
    )


    print(
        f"\n--- ROUTING: "
        f"{parent_column}=1 -> {child_column} ---"
    )

    print(
        "Expected populated:",
        int(
            expected.sum()
        )
    )

    print(
        "Actual populated:",
        int(
            actual.sum()
        )
    )

    print(
        "Mismatches:",
        mismatches
    )


    if mismatches != 0:

        raise ValueError(
            f"Routing failed for "
            f"{parent_column} -> {child_column}."
        )


    print(
        "PASS: Routing validated."
    )


# ============================================================
# 24. ROUTING VALIDATIONS
# ============================================================

check_yes_route(
    "PHQ020",
    "PHACOFHR"
)

check_yes_route(
    "PHQ020",
    "PHACOFMN"
)


check_yes_route(
    "PHQ030",
    "PHAALCHR"
)

check_yes_route(
    "PHQ030",
    "PHAALCMN"
)


check_yes_route(
    "PHQ040",
    "PHAGUMHR"
)

check_yes_route(
    "PHQ040",
    "PHAGUMMN"
)


check_yes_route(
    "PHQ050",
    "PHAANTHR"
)

check_yes_route(
    "PHQ050",
    "PHAANTMN"
)


check_yes_route(
    "PHQ060",
    "PHASUPHR"
)

check_yes_route(
    "PHQ060",
    "PHASUPMN"
)


# ============================================================
# 25. PHQ MISSING-POSITION CONSISTENCY
# ============================================================

#
# All five Yes/No fasting questions have:
#
#     7,996 populated
#     370 missing
#
# Their missing masks should match.
#

QUESTION_COLUMNS = [
    "PHQ020",
    "PHQ030",
    "PHQ040",
    "PHQ050",
    "PHQ060"
]


base_missing = (
    df["PHQ020"]
    .isna()
    .to_numpy()
)


print(
    "\n--- PHQ MISSING-POSITION ALIGNMENT ---"
)


for col in QUESTION_COLUMNS[1:]:

    mismatch_count = int(
        np.sum(
            base_missing
            !=
            df[col]
            .isna()
            .to_numpy()
        )
    )


    print(
        f"PHQ020 vs {col}:",
        mismatch_count
    )


    if mismatch_count != 0:

        raise ValueError(
            f"{col} missing positions differ "
            "from PHQ020."
        )


print(
    "PASS: All PHQ question missing masks match."
)


# ============================================================
# 26. TOTAL FASTING-TIME MISSING ALIGNMENT
# ============================================================

#
# PHAFSTHR and PHAFSTMN must be populated/missing together.
#

fast_time_missing_mismatch = int(
    np.sum(
        df["PHAFSTHR"]
        .isna()
        .to_numpy()
        !=
        df["PHAFSTMN"]
        .isna()
        .to_numpy()
    )
)


print(
    "\n--- TOTAL FASTING-TIME ALIGNMENT ---"
)

print(
    "PHAFSTHR vs PHAFSTMN missing mismatches:",
    fast_time_missing_mismatch
)


if fast_time_missing_mismatch != 0:

    raise ValueError(
        "PHAFSTHR/PHAFSTMN missing "
        "positions differ."
    )


print(
    "PASS: Total fasting hour/minute "
    "missing positions match."
)


# ============================================================
# 27. FINAL ZERO PATTERN BEFORE Int64
# ============================================================

#
# After repair, the only zeros should correspond exactly
# to the validated tiny-zero counts.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if count > 0:

        final_zero_counts[col] = count


print(
    "\n--- FINAL ZERO PATTERN ---"
)

print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs from "
        "validated FASTQX_J pattern.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


print(
    "PASS: Final zero pattern validated."
)


# ============================================================
# 28. PRE-CONVERSION NUMERIC FIDELITY
# ============================================================

#
# The ONLY allowed changes:
#
#     validated tiny XPORT values -> 0
#
# Nothing else may change.
#

print(
    "\n--- PRE-CONVERSION NUMERIC FIDELITY ---"
)


pre_conversion_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_conversion_failures.append(
            col
        )


if pre_conversion_failures:

    raise ValueError(
        "Unexpected numeric changes before Int64 "
        "conversion:\n"
        + str(pre_conversion_failures)
    )


print(
    "PASS: Corrected dataframe exactly matches "
    "expected XPT-derived values."
)


# ============================================================
# 29. CONVERT ALL PROVEN INTEGER FIELDS TO Int64
# ============================================================

#
# No .round() transformation is used.
#
# Every value has already been proven integral.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if not whole_mask.all():

        raise ValueError(
            f"{col} unexpectedly contains "
            "fractional values."
        )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 19 FASTQX_J fields "
    "converted to nullable Int64."
)


# ============================================================
# 30. PRE-EXPORT NUMERIC FIDELITY
# ============================================================

print(
    "\n--- PRE-EXPORT NUMERIC FIDELITY ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(pre_export_failures)
    )


print(
    "PASS: All 19 variables match exact "
    "expected corrected values."
)


# ============================================================
# 31. MISSING-POSITION FIDELITY BEFORE EXPORT
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    source_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    current_missing = (
        df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            source_missing
            !=
            current_missing
        )
    )


print(
    "\nMissing-position mismatches before export:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 32. EXPORT CSV
# ============================================================

#
# CRITICAL NHANES FIDELITY POLICY
# ------------------------------------------------------------
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
# NO:
#
#     blanket decimal formatting
#
#
# FASTQX_J happens to contain only whole-number fields,
# but this export rule is intentionally compatible with
# mixed NHANES files containing genuine decimals.
#
# Genuine Dietary/Lab/Exam/Weight decimal variables in
# other files should remain numerically AS IS.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 33. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

#
# Use the same corrected approach established for
# TRIGLY_J and the other lab files.
#

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 34. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV fidelity differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 19 variables round-trip exactly."
)


# ============================================================
# 35. CSV MISSING-VALUE FIDELITY
# ============================================================

csv_missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    csv_missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- CSV MISSING-VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    csv_missing_mismatches
)


if csv_missing_mismatches != 0:

    raise ValueError(
        "Missing positions changed during CSV export."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 36. READ FINAL CSV AS RAW TEXT
# ============================================================

#
# Raw text is used for formatting checks only.
#

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


if raw_csv_df.shape != (
    8366,
    19
):

    raise ValueError(
        "Raw CSV dimensions changed."
    )


if (
    raw_csv_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "Raw CSV column order changed."
    )


# ============================================================
# 37. RAW CSV .0 CHECK
# ============================================================

#
# All 19 FASTQX_J variables were proven integral.
#
# Therefore ALL may safely be checked for unnecessary ".0".
#

print(
    "\n--- RAW CSV .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' "
    "in any FASTQX_J field."
)


# ============================================================
# 38. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY-VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 39. FINAL CSV ZERO PATTERN
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    count = int(
        values.eq(0)
        .sum()
    )


    if count > 0:

        csv_zero_counts[col] = count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV zero pattern "
    "matches validated source corrections."
)


# ============================================================
# 40. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_errors = {}


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    count = int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


    if count > 0:

        scientific_errors[col] = count


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    scientific_errors
)


if scientific_errors:

    raise ValueError(
        "Unexpected scientific notation found:\n"
        + str(scientific_errors)
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 41. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 93703:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 102956:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.isna()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains missing SEQN."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN 93703-102956 valid "
    "and contains no '.0'."
)


# ============================================================
# 42. FINAL ALL-COLUMN SOURCE -> CSV FIDELITY
# ============================================================

#
# Use roundtrip_df for exact numeric comparisons.
#
# Do NOT reparse raw strings for exact binary-float
# equality.
#

print(
    "\n--- FINAL ALL-COLUMN SOURCE -> CSV FIDELITY ---"
)


final_fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    final_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        final_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        final_fidelity_failures.append(
            col
        )


if final_fidelity_failures:

    raise ValueError(
        "Final source -> CSV fidelity failures:\n"
        + str(final_fidelity_failures)
    )


print(
    "PASS: Final CSV preserves all expected "
    "FASTQX_J numeric values exactly."
)


# ============================================================
# 43. ROUND-TRIP MAXIMUM DIFFERENCE DIAGNOSTIC
# ============================================================

#
# Expected:
#
#     0.0
#
# for all variables.
#

print(
    "\n--- ROUND-TRIP MAXIMUM ABSOLUTE DIFFERENCES ---"
)


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    final_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    valid_mask = (
        ~np.isnan(expected_values)
        &
        ~np.isnan(final_values)
    )


    if valid_mask.any():

        max_difference = float(
            np.max(
                np.abs(
                    expected_values[valid_mask]
                    -
                    final_values[valid_mask]
                )
            )
        )

    else:

        max_difference = 0.0


    print(
        f"{col}: {max_difference}"
    )


# ============================================================
# 44. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "FASTQX_J CONVERSION COMPLETE"
)

print(
    "=" * 90
)

print(
    "Cycle: 2017-2018"
)

print(
    "Component: Laboratory - Fasting Questionnaire"
)

print(
    "Eligible population: Examined participants age 1+"
)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)

print(
    "Original tiny XPORT artifacts:",
    tiny_total
)

print(
    "Validated zero restorations:",
    total_corrections
)

print(
    "Variables containing restored zeros:",
    len(EXPECTED_TINY_COUNTS)
)

print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)

print(
    "Morning session:",
    int(
        df["PHDSESN"]
        .eq(0)
        .sum()
    )
)

print(
    "Afternoon session:",
    int(
        df["PHDSESN"]
        .eq(1)
        .sum()
    )
)

print(
    "Evening session:",
    int(
        df["PHDSESN"]
        .eq(2)
        .sum()
    )
)

print(
    "PHAFSTHR populated:",
    int(
        df["PHAFSTHR"]
        .notna()
        .sum()
    )
)

print(
    "PHAFSTMN populated:",
    int(
        df["PHAFSTMN"]
        .notna()
        .sum()
    )
)

print(
    "Exact XPT -> CSV fidelity failures: "
    "0 expected"
)

print(
    "Missing-position differences: "
    "0 expected"
)

print(
    "Tiny values remaining: "
    "0 expected"
)

print(
    "Unnecessary '.0': "
    "0 expected"
)

print(
    "Scientific notation: "
    "0 expected"
)

print(
    "=" * 90
)

NHANES 2017-2018 FASTQX_J XPT -> CSV
Rows: 8,366
Columns: 19
PASS: Structure = 8,366 rows x 19 columns.

--- FIELD TYPE PLAN ---
Whole-number/code fields: 19
Genuine decimal fields: []
PASS: All 19 FASTQX_J variables loaded numerically.

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before repair: 0
Tiny XPORT artifacts: 4,389

Tiny values by variable:
PHDSESN     3818
PHAFSTHR     325
PHAFSTMN     146
PHAGUMHR      63
PHAGUMMN       8
PHAANTHR       7
PHASUPHR       5
PHASUPMN       5
PHACOFHR       5
PHACOFMN       2
PHAALCMN       2
PHAANTMN       2
PHAALCHR       1
PASS: Exact 4,389-value tiny-zero pattern validated.

--- VALIDATED ZERO RESTORATION ---
PHACOFHR: 5
PHACOFMN: 2
PHAALCHR: 1
PHAALCMN: 2
PHAGUMHR: 63
PHAGUMMN: 8
PHAANTHR: 7
PHAANTMN: 2
PHASUPHR: 5
PHASUPMN: 5
PHAFSTHR: 325
PHAFSTMN: 146
PHDSESN: 3,818
PASS: 4,389 validated XPORT zeros restored.

Tiny artifacts remaining: 0
PASS: No tiny XPORT artifacts remain.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional 